# resnet-stem — ex1: build the ResNet stem and verify the 224 → 56 reduction

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `resnet-stem`. Running the final beacon cell reports progress against the `CNN: ResNet stem block` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ResNet stem block` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`resnet-stem`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "resnet-stem"
DD_SUBTOPIC = "CNN: ResNet stem block"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ResNet stem block — quick refresher

The 'stem' is the FIRST stage of every ResNet — the four-op sequence that turns a `(B, 3, 224, 224)` ImageNet image into a `(B, 64, 56, 56)` feature map ready for the first BlockGroup:

```
nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
    nn.BatchNorm2d(64),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
)
```

**Shape walkthrough (224 input):**
- Input:           `(B, 3, 224, 224)`
- After 7×7 s=2 p=3 conv: `(B, 64, 112, 112)`  → `(224 + 2*3 - 7) // 2 + 1 = 112`
- After BN + ReLU: `(B, 64, 112, 112)` (no shape change)
- After 3×3 s=2 p=1 MaxPool: `(B, 64, 56, 56)` → `(112 + 2*1 - 3) // 2 + 1 = 56`

**Why this exact recipe.**
- **7×7 conv with stride 2** — large receptive field early, halves spatial size cheaply. The wide kernel captures edge/blob features that small kernels need multiple layers to see.
- **`bias=False`** — pointless to learn a bias right before BatchNorm (which subtracts the mean and learns its own bias `beta`). Saves 64 params and a tiny amount of compute.
- **BatchNorm + ReLU** — the canonical 'conv block' suffix. Same pattern repeats inside every ResidualBlock.
- **MaxPool with stride 2** — second 2× spatial reduction. Combined with the conv stride, the stem downsamples by 4× in total (224 → 56) before any BlockGroup runs.

**Why subsequent BlockGroups don't need a 'stem'.** Each BlockGroup's FIRST ResidualBlock has `stride=2` (except group 0), which performs the per-group 2× downsample. The stem is special because it has to go from 3 channels → 64 in one shot, with a big enough kernel to see meaningful image structure.

### Exercise 1 — build the ResNet stem and verify the 224 → 56 reduction

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the ResNet stem recipe — Conv(7×7, s=2, p=3) + BN + ReLU + MaxPool(3×3, s=2, p=1) — wired into an `nn.Sequential` that turns `(B, 3, 224, 224)` into `(B, 64, 56, 56)`.
> Keywords: resnet, stem, downsample, sequential, 224-to-56
> ```

**KCs targeted:** `resnet-stem-recipe`, `stem-shape-reduction-math`

Implement `ex1_build_resnet_stem()` that returns an `nn.Sequential` containing the four-op ResNet stem block, in this exact order:

1. `nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False)`
2. `nn.BatchNorm2d(64)`
3. `nn.ReLU(inplace=True)`
4. `nn.MaxPool2d(kernel_size=3, stride=2, padding=1)`

**Why each setting.**
- `kernel_size=7, stride=2, padding=3` on the conv → big receptive field early, halves spatial size from 224 to 112. Shape: `(224 + 2*3 - 7) // 2 + 1 = 112`.
- `bias=False` on the conv → bias is redundant before a BatchNorm that already has a learnable `beta` shift.
- `inplace=True` on the ReLU → micro-optimization; the conv output isn't used elsewhere so overwriting it is safe.
- `kernel_size=3, stride=2, padding=1` on the MaxPool → second 2× spatial reduction. Shape: `(112 + 2*1 - 3) // 2 + 1 = 56`.

Putting it all together: `(B, 3, 224, 224) → (B, 64, 112, 112) → ... → (B, 64, 56, 56)`. That's a 4× total spatial downsample plus a 3 → 64 channel lift in one block.

**What the test checks.**
- The returned object is `nn.Sequential` with 4 children.
- Child types are correct, in the right order.
- The conv has `kernel_size=7, stride=2, padding=3, bias=None`.
- The maxpool has `kernel_size=3, stride=2, padding=1`.
- Forward on `(B, 3, 224, 224)` yields exactly `(B, 64, 56, 56)`.
- Smaller input (`(B, 3, 64, 64)`) yields `(B, 64, 16, 16)` — the 4× downsample is general, not 224-specific.

In [ ]:
def ex1_build_resnet_stem():
    import torch.nn as nn
    return nn.Sequential(
        nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    )


<details><summary>Solution</summary>

```python
def ex1_build_resnet_stem():
    import torch.nn as nn
    return nn.Sequential(
        nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    )
```

**Why this exact spatial math.** The two stride-2 layers together produce a 4× downsample. Pairing the conv's `stride=2` with `padding=(K-1)/2 = 3` gives a CLEAN halving (not (224+5)//2+1=114, but (224+6-7)//2+1=112). The same trick applies to the MaxPool: `K=3, P=1` halves cleanly.

**Why `bias=False` AND BatchNorm.** Both BN and bias add a per-channel constant. Stacking them is wasteful — BN's `beta` is exactly the same expressive power. ResNet sets every conv with a following BN to `bias=False` for this reason. The 64 saved scalars don't matter — but the PRINCIPLE matters every time you compose conv + norm.

**Why MaxPool here, AvgPool at the head.** Stem MaxPool selects salient activations (after the first conv has produced edge/blob features). Head AvgPool (between last BlockGroup and `fc`) averages — that's what a classifier wants. Both share the einops `'b c (h p1) (w p2) -> b c h w'` axis-factor pattern; only the reducer differs.

**Real torchvision stem.** `torchvision.models.resnet18().conv1` + `.bn1` + `.relu` + `.maxpool` matches this exactly. You can `assert str(stem) == str(resnet18_stem)` against the official model if you want to confirm the recipe.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()